In [1]:
# ════════════════════════════════════════════════════════
# CELL 1 — IMPORTS & DATA LOADING
# ════════════════════════════════════════════════════════
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack, csr_matrix
import warnings
warnings.filterwarnings('ignore')

print('=' * 55)
print('   DIAPILOT — Experiment 1: KNN Baseline v2')
print('=' * 55)

df = pd.read_parquet('../data/processed/DiaPilot_Combined_Data.parquet')
print(f'Dataset loaded: {len(df):,} recipes')

   DIAPILOT — Experiment 1: KNN Baseline v2
Dataset loaded: 117,662 recipes


In [2]:
# ════════════════════════════════════════════════════════
# CELL 2 — MEDICAL ENGINE (ADA-ALIGNED)
# Fix: Sugar = clinical tier (8g/15g), not daily_sugar/3
# ════════════════════════════════════════════════════════

def calculate_tdee(age, weight_kg, height_cm, gender, activity='sedentary'):
    if gender.lower() == 'male':
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5
    else:
        bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) - 161
    multipliers = {'sedentary':1.2,'light':1.375,'moderate':1.55,'active':1.725}
    return round(bmr * multipliers.get(activity, 1.2))


def get_dynamic_macros(tdee, leicester_score, on_medication=False,
                        current_glucose_high=False):
    """
    ADA-aligned per-meal constraints.
    Sugar is a clinical tier, NOT daily_sugar_g / 3.
    """
    if leicester_score <= 10:
        carb_pct, sugar_limit = 0.50, 15.0
    elif leicester_score <= 15:
        carb_pct, sugar_limit = 0.40, 10.0
    else:
        carb_pct, sugar_limit = 0.30,  8.0

    if current_glucose_high:
        carb_pct   -= 0.10
        sugar_limit =  3.0

    per_meal_carbs_max = round(((tdee * carb_pct) / 4) / 3)
    per_meal_carbs_min = 25 if on_medication else 0

    if on_medication and per_meal_carbs_max < 25:
        per_meal_carbs_max = 25

    return {
        'per_meal_calories'  : round(tdee / 3),
        'per_meal_max_carbs' : per_meal_carbs_max,
        'per_meal_min_carbs' : per_meal_carbs_min,
        'per_meal_sugar'     : sugar_limit
    }


# ── Patient profile ─────────────────────────────────────
PATIENT_AGE       = 45
PATIENT_WEIGHT_KG = 85
PATIENT_HEIGHT_CM = 175
PATIENT_GENDER    = 'male'
PATIENT_ACTIVITY  = 'sedentary'
LEICESTER_SCORE   = 14
ON_MEDICATION     = True
GLUCOSE_HIGH      = False

patient_tdee   = calculate_tdee(PATIENT_AGE, PATIENT_WEIGHT_KG,
                                 PATIENT_HEIGHT_CM, PATIENT_GENDER,
                                 PATIENT_ACTIVITY)
patient_macros = get_dynamic_macros(patient_tdee, LEICESTER_SCORE,
                                     ON_MEDICATION, GLUCOSE_HIGH)

print('Medical Engine — ADA 2025 Aligned')
print('-' * 40)
print(f'  TDEE             : {patient_tdee} kcal/day')
print(f'  Per-meal calories: {patient_macros["per_meal_calories"]} kcal')
print(f'  Carb window      : {patient_macros["per_meal_min_carbs"]}g'
      f' - {patient_macros["per_meal_max_carbs"]}g')
print(f'  Sugar max        : {patient_macros["per_meal_sugar"]}g')
print('Medical engine ready.')

Medical Engine — ADA 2025 Aligned
----------------------------------------
  TDEE             : 2068 kcal/day
  Per-meal calories: 689 kcal
  Carb window      : 25g - 69g
  Sugar max        : 10.0g
Medical engine ready.


In [3]:
# ════════════════════════════════════════════════════════
# CELL 3 — MEAL TYPE CLASSIFICATION
# Fix: classify every recipe before filtering
# ════════════════════════════════════════════════════════

BREAKFAST_KW = ['egg','oat','oatmeal','paratha','halwa','dahi','yogurt',
                'pancake','toast','cereal','porridge','smoothie','muffin',
                'waffle','granola','scramble','omelette','crepe']

DINNER_KW    = ['karahi','biryani','gosht','daal','curry','sabzi','stew',
                'roast','steak','bake','casserole','kebab','tikka','nihari',
                'korma','palak','keema','chops','grilled chicken','lamb']

def classify_meal_type(row):
    text    = (str(row['Recipe_Name']) + ' ' + str(row['Ingredients'])).lower()
    b_score = sum(1 for kw in BREAKFAST_KW if kw in text)
    d_score = sum(1 for kw in DINNER_KW   if kw in text)
    if b_score > d_score: return 'Breakfast'
    elif d_score > 0:     return 'Dinner'
    else:                 return 'Lunch'

df['Meal_Type'] = df.apply(classify_meal_type, axis=1)
counts = df['Meal_Type'].value_counts()
print('Meal type classification complete')
print('-' * 40)
for mt in ['Breakfast','Lunch','Dinner']:
    print(f'  {mt:12s}: {counts.get(mt,0):,} recipes')

Meal type classification complete
----------------------------------------
  Breakfast   : 28,429 recipes
  Lunch       : 63,925 recipes
  Dinner      : 25,308 recipes


In [4]:
# ════════════════════════════════════════════════════════
# CELL 4 — MEDICAL FILTER (no NLP regional filter)
# Key difference from v1: full global pool given to KNN
# Regional filtering is Gemini's job, not KNN's job
# ════════════════════════════════════════════════════════

safe_df = df[
    (df['Calories'] <= patient_macros['per_meal_calories']) &
    (df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (df['Sugar_g']  <= patient_macros['per_meal_sugar'])
].copy().reset_index(drop=True)

print(f'Medically safe pool: {len(safe_df):,} recipes')
safe_counts = safe_df['Meal_Type'].value_counts()
print('-' * 40)
for mt in ['Breakfast','Lunch','Dinner']:
    print(f'  {mt:12s}: {safe_counts.get(mt,0):,} recipes')

if len(safe_df) < 90:
    print(f'WARNING: Only {len(safe_df)} safe recipes. '
          f'Plan will cycle through available meals.')

Medically safe pool: 21,763 recipes
----------------------------------------
  Breakfast   : 4,325 recipes
  Lunch       : 13,228 recipes
  Dinner      : 4,210 recipes


In [5]:
# ════════════════════════════════════════════════════════
# CELL 5 — KNN FEATURE EXTRACTION WITH CLINICAL WEIGHTS
# Fix: Protein added · Carbs 2.5x · Sugar 2.0x · Nutrition 3x over TF-IDF
# ════════════════════════════════════════════════════════

def build_feature_matrix(subset_df):
    """
    Builds a weighted feature matrix for KNN.
    Nutrition features get 3x weight over ingredient text
    because medical safety depends on macros, not ingredient names.
    Within nutrition: Carbs 2.5x, Sugar 2.0x (highest glycaemic impact).
    """
    # Text features
    vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
    X_text     = vectorizer.fit_transform(subset_df['Ingredients'])

    # Numerical features: [Calories, Carbs_g, Sugar_g, Protein_g]
    scaler  = MinMaxScaler()
    X_num   = scaler.fit_transform(
        subset_df[['Calories','Carbs_g','Sugar_g','Protein_g']]
    )

    # Clinical importance weights
    clinical_weights        = np.array([1.0, 2.5, 2.0, 0.8])
    X_num_weighted          = X_num * clinical_weights
    X_num_weighted         *= 3.0   # nutrition block 3x over TF-IDF

    X_combined = hstack([X_text, X_num_weighted])
    return X_combined, vectorizer, scaler


def train_knn(X_combined, n_neighbors=30):
    k   = min(n_neighbors, X_combined.shape[0])
    knn = NearestNeighbors(n_neighbors=k, metric='cosine', algorithm='brute')
    knn.fit(X_combined)
    return knn


# Train one KNN per meal type for clinical appropriateness
knn_models = {}
feat_matrices = {}
meal_dfs = {}

for meal_type in ['Breakfast', 'Lunch', 'Dinner']:
    pool = safe_df[safe_df['Meal_Type'] == meal_type].copy().reset_index(drop=True)

    if len(pool) < 5:
        print(f'  WARNING: {meal_type} pool only {len(pool)} recipes'
              f' — using full safe pool as fallback')
        pool = safe_df.copy().reset_index(drop=True)

    X, vec, scl       = build_feature_matrix(pool)
    knn                = train_knn(X, n_neighbors=30)
    knn_models[meal_type]  = knn
    feat_matrices[meal_type] = X
    meal_dfs[meal_type]      = pool
    print(f'  KNN trained for {meal_type:12s}: {len(pool):,} recipes, '
          f'{X.shape[1]} features')

print('All KNN models ready.')

  KNN trained for Breakfast   : 4,325 recipes, 504 features
  KNN trained for Lunch       : 13,228 recipes, 504 features
  KNN trained for Dinner      : 4,210 recipes, 504 features
All KNN models ready.


In [6]:
# ════════════════════════════════════════════════════════
# CELL 6 — MMR DIVERSITY SELECTION
# Fix: 4-macro similarity (not single-column)
# Fix: Applied to ALL meal types including Breakfast
# ════════════════════════════════════════════════════════

def apply_mmr_knn(candidate_df, feature_matrix, knn_model,
                  top_k=30, lambda_param=0.7):
    """
    KNN-specific MMR:
      - Relevance  = inverse KNN distance from anchor (row 0)
      - Diversity  = nutritional dissimilarity across 4 macros
      - lambda=0.7 → 70% relevance, 30% diversity
    """
    n = len(candidate_df)
    if n == 0:
        return pd.DataFrame()
    if n <= top_k:
        return candidate_df.reset_index(drop=True)

    # Get KNN distances from anchor (first recipe in pool)
    anchor      = feature_matrix.getrow(0)
    k_query     = min(n, feature_matrix.shape[0])
    distances, indices = knn_model.kneighbors(anchor, n_neighbors=k_query)
    dist_map    = {idx: dist for idx, dist in
                   zip(indices[0], distances[0])}

    # Normalise macro columns for distance comparison
    macro_cols  = ['Calories','Carbs_g','Sugar_g','Protein_g']
    macro_vals  = candidate_df[macro_cols].values.astype(float)
    col_range   = macro_vals.max(axis=0) - macro_vals.min(axis=0)
    col_range   = np.where(col_range > 0, col_range, 1.0)
    macro_norm  = (macro_vals - macro_vals.min(axis=0)) / col_range

    remaining   = list(range(n))
    selected    = []

    # Start with most relevant (lowest KNN distance)
    best_start  = min(remaining,
                      key=lambda i: dist_map.get(i, float('inf')))
    selected.append(best_start)
    remaining.remove(best_start)

    while len(selected) < top_k and remaining:
        best_mmr = -float('inf')
        best_idx = None

        for i in remaining:
            relevance = 1.0 / (1.0 + dist_map.get(i, 1.0))
            sims      = [
                1.0 / (1.0 + np.linalg.norm(macro_norm[i] - macro_norm[s]))
                for s in selected
            ]
            max_sim   = max(sims)
            mmr       = lambda_param * relevance - (1 - lambda_param) * max_sim

            if mmr > best_mmr:
                best_mmr = mmr
                best_idx = i

        selected.append(best_idx)
        remaining.remove(best_idx)

    return candidate_df.iloc[selected].reset_index(drop=True)


print('MMR function defined.')
print('Similarity: 4-macro Euclidean (Calories, Carbs, Sugar, Protein)')

MMR function defined.
Similarity: 4-macro Euclidean (Calories, Carbs, Sugar, Protein)


In [7]:
# ════════════════════════════════════════════════════════
# CELL 7 — GENERATE COMPLETE 30-DAY PLAN
# Fix: 90 meals total (30 days x 3 slots)
# Fix: Per-slot MMR, not 3-anchor rotation
# ════════════════════════════════════════════════════════

DAYS_IN_PLAN = 30
MEAL_SLOTS   = ['Breakfast', 'Lunch', 'Dinner']

plan_rows    = []
pool_summary = {}

for meal_type in MEAL_SLOTS:
    pool    = meal_dfs[meal_type]
    X       = feat_matrices[meal_type]
    knn     = knn_models[meal_type]

    selected = apply_mmr_knn(pool, X, knn,
                              top_k=DAYS_IN_PLAN, lambda_param=0.7)
    pool_summary[meal_type] = len(selected)

    for day_idx in range(DAYS_IN_PLAN):
        row = selected.iloc[day_idx % len(selected)]
        plan_rows.append({
            'Day'      : f'Day {day_idx + 1}',
            'Day_Num'  : day_idx + 1,
            'Meal'     : meal_type,
            'Recipe'   : row['Recipe_Name'],
            'Calories' : round(row['Calories'],  1),
            'Carbs_g'  : round(row['Carbs_g'],   1),
            'Sugar_g'  : round(row['Sugar_g'],   1),
            'Protein_g': round(row['Protein_g'], 1),
        })

meal_order    = {'Breakfast':0,'Lunch':1,'Dinner':2}
plan_df       = pd.DataFrame(plan_rows)
plan_df['Meal_Order'] = plan_df['Meal'].map(meal_order)
plan_df       = (
    plan_df
    .sort_values(['Day_Num','Meal_Order'])
    .drop(columns=['Day_Num','Meal_Order'])
    .reset_index(drop=True)
)

print('30-Day KNN Diet Plan Generated')
print('=' * 55)
print(f'  Total meals : {len(plan_df)}  (30 days x 3 slots)')
for mt, count in pool_summary.items():
    print(f'  {mt:12s}: {count} unique meals selected by MMR')
print()
print('Sample — First 9 meals:')
print(plan_df[['Day','Meal','Recipe','Calories','Carbs_g','Sugar_g']]
      .head(9).to_string(index=False))

30-Day KNN Diet Plan Generated
  Total meals : 90  (30 days x 3 slots)
  Breakfast   : 30 unique meals selected by MMR
  Lunch       : 30 unique meals selected by MMR
  Dinner      : 30 unique meals selected by MMR

Sample — First 9 meals:
  Day      Meal                                      Recipe  Calories  Carbs_g  Sugar_g
Day 1 Breakfast                     crispy crunchy  chicken     335.8     27.5      1.0
Day 1     Lunch                          alouette  potatoes     368.1     55.0      5.0
Day 1    Dinner                       forgotten  minestrone     346.9     27.5      9.0
Day 2 Breakfast                       light chicken nuggets     676.0     27.5      3.5
Day 2     Lunch              new orleans red beans and rice     608.1     68.8      8.5
Day 2    Dinner                     grilled chicken burrito     684.7     27.5      9.0
Day 3 Breakfast chicken stuffed with goat cheese and garlic     469.8     27.5      2.5
Day 3     Lunch                            soft burger b

In [8]:
# ════════════════════════════════════════════════════════
# CELL 8 — EVALUATION
# ════════════════════════════════════════════════════════

print('=' * 55)
print('   KNN BASELINE v2 — EVALUATION REPORT')
print('=' * 55)

# Medical compliance
compliant = plan_df[
    (plan_df['Carbs_g']  <= patient_macros['per_meal_max_carbs']) &
    (plan_df['Carbs_g']  >= patient_macros['per_meal_min_carbs']) &
    (plan_df['Sugar_g']  <= patient_macros['per_meal_sugar'])     &
    (plan_df['Calories'] <= patient_macros['per_meal_calories'])
]
compliance = (len(compliant) / len(plan_df)) * 100

print(f'\n1. Medical Compliance')
print(f'   ADA-safe meals : {len(compliant)} / {len(plan_df)}')
print(f'   Rate           : {compliance:.1f}%')

print(f'\n2. Nutritional Profile (plan average per meal)')
print(f'   Avg Calories   : {plan_df["Calories"].mean():.1f} kcal')
print(f'   Avg Carbs      : {plan_df["Carbs_g"].mean():.1f}g '
      f'(target: {patient_macros["per_meal_min_carbs"]}-'
      f'{patient_macros["per_meal_max_carbs"]}g)')
print(f'   Avg Sugar      : {plan_df["Sugar_g"].mean():.1f}g '
      f'(limit: {patient_macros["per_meal_sugar"]}g)')
print(f'   Avg Protein    : {plan_df["Protein_g"].mean():.1f}g')

unique_meals = plan_df['Recipe'].nunique()
variety_pct  = (unique_meals / len(plan_df)) * 100
print(f'\n3. Variety Score')
print(f'   Unique recipes : {unique_meals} / {len(plan_df)}')
print(f'   Variety        : {variety_pct:.1f}%')

print(f'\n4. Per-Slot Variety')
for mt in MEAL_SLOTS:
    slot_df   = plan_df[plan_df['Meal'] == mt]
    slot_uniq = slot_df['Recipe'].nunique()
    print(f'   {mt:12s}: {slot_uniq} unique / 30 days')

print(f'\n5. ADA Range Check (45-60g carbs per meal)')
in_ada = plan_df[(plan_df['Carbs_g'] >= 45) & (plan_df['Carbs_g'] <= 60)]
print(f'   In ADA range   : {len(in_ada)} / {len(plan_df)} meals')

print('\n' + '=' * 55)

   KNN BASELINE v2 — EVALUATION REPORT

1. Medical Compliance
   ADA-safe meals : 90 / 90
   Rate           : 100.0%

2. Nutritional Profile (plan average per meal)
   Avg Calories   : 476.7 kcal
   Avg Carbs      : 40.1g (target: 25-69g)
   Avg Sugar      : 6.0g (limit: 10.0g)
   Avg Protein    : 30.3g

3. Variety Score
   Unique recipes : 90 / 90
   Variety        : 100.0%

4. Per-Slot Variety
   Breakfast   : 30 unique / 30 days
   Lunch       : 30 unique / 30 days
   Dinner      : 30 unique / 30 days

5. ADA Range Check (45-60g carbs per meal)
   In ADA range   : 9 / 90 meals



In [9]:
# ════════════════════════════════════════════════════════
# CELL 9 — PDF REPORT (named KNN Diet Plan)
# ════════════════════════════════════════════════════════

from fpdf import FPDF
import os

def to_safe(text, limit=40):
    """Strip non-latin-1 chars so fpdf never crashes."""
    return str(text).encode('latin-1', errors='replace').decode('latin-1')[:limit].title()


class KnnPDF(FPDF):
    def header(self):
        self.set_font('Arial', 'B', 13)
        self.set_text_color(29, 158, 117)
        self.cell(0, 10, 'DIAPILOT - KNN Diet Plan', ln=True, align='C')
        self.set_font('Arial', 'I', 9)
        self.set_text_color(100, 100, 100)
        self.cell(0, 6,
                  'Experiment 1: KNN Baseline v2 | Cosine Similarity + MMR | ADA 2025',
                  ln=True, align='C')
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(3)

    def footer(self):
        self.set_y(-15)
        self.set_font('Arial', 'I', 8)
        self.set_text_color(150, 150, 150)
        self.cell(0, 10,
                  f'Page {self.page_no()} | Diapilot KNN Baseline v2',
                  align='C')


pdf = KnnPDF()
pdf.set_auto_page_break(auto=True, margin=20)
pdf.add_page()

# Patient profile box
pdf.set_font('Arial', 'B', 9)
pdf.set_fill_color(225, 245, 238)
pdf.set_text_color(8, 80, 65)
pdf.cell(0, 8,
         f'Patient | TDEE: {patient_tdee} kcal | '
         f'Leicester: {LEICESTER_SCORE} | '
         f'Medication: {"Yes" if ON_MEDICATION else "No"} | '
         f'Carbs: {patient_macros["per_meal_min_carbs"]}'
         f'-{patient_macros["per_meal_max_carbs"]}g | '
         f'Sugar: max {patient_macros["per_meal_sugar"]}g',
         border=1, ln=True, fill=True, align='C')
pdf.ln(3)

# Table header
col_widths = [18, 25, 75, 20, 18, 18, 16]
headers    = ['Day','Meal','Recipe Name','Cal','Carbs','Sugar','Pro']
pdf.set_font('Arial', 'B', 9)
pdf.set_fill_color(29, 158, 117)
pdf.set_text_color(255, 255, 255)
for h, w in zip(headers, col_widths):
    pdf.cell(w, 8, h, border=1, align='C', fill=True)
pdf.ln()

meal_colors = {
    'Breakfast': (255, 253, 240),
    'Lunch'    : (240, 248, 255),
    'Dinner'   : (245, 240, 255),
}
pdf.set_font('Arial', '', 8)
current_day = ''

for _, row in plan_df.iterrows():
    r, g, b = meal_colors[row['Meal']]
    pdf.set_fill_color(r, g, b)
    pdf.set_text_color(30, 30, 30)

    day_label   = row['Day'] if row['Day'] != current_day else ''
    current_day = row['Day']

    pdf.cell(col_widths[0], 7, day_label,                   border=1, align='C', fill=True)
    pdf.cell(col_widths[1], 7, row['Meal'],                  border=1, align='C', fill=True)
    pdf.cell(col_widths[2], 7, to_safe(row['Recipe'], 38),   border=1,            fill=True)
    pdf.cell(col_widths[3], 7, str(row['Calories']),         border=1, align='C', fill=True)
    pdf.cell(col_widths[4], 7, f"{row['Carbs_g']}g",        border=1, align='C', fill=True)
    pdf.cell(col_widths[5], 7, f"{row['Sugar_g']}g",        border=1, align='C', fill=True)
    pdf.cell(col_widths[6], 7, f"{row['Protein_g']}g",      border=1, align='C', fill=True)
    pdf.ln()

os.makedirs('../reports', exist_ok=True)
pdf.output('../reports/Diapilot_KNN_Diet_Plan.pdf')

plan_df.to_csv('../reports/Diapilot_KNN_Diet_Plan.csv', index=False)

print('PDF saved: ../reports/Diapilot_KNN_Diet_Plan.pdf')
print('CSV saved: ../reports/Diapilot_KNN_Diet_Plan.csv')
print(f'Total meals in report: {len(plan_df)}')

PDF saved: ../reports/Diapilot_KNN_Diet_Plan.pdf
CSV saved: ../reports/Diapilot_KNN_Diet_Plan.csv
Total meals in report: 90
